# notebook で metric を開発して、パッケージに組み込む

新しい評価指標（metric）を notebook で試作し、`alpha_signal_evaluation` パッケージに組み込むまでを順に説明します。
上から順に実行すれば、最後まで動きます（合成データを使うので、手元にデータがなくても実行できます）。

**題材: 上位 N 銘柄の超過リターン（`top_n_excess`）**
日付ごとに、シグナル上位 N 銘柄のフォワードリターンの等ウェイト平均から、全銘柄の平均を引いたもの。

| Step | やること | 作業する場所 |
|---|---|---|
| 1 | 準備 | notebook |
| 2 | 評価データ `DataBundle` を触ってみる | notebook |
| 3 | pandas で計算ロジックを試作する | notebook |
| 4 | `Metric` クラスにする | notebook |
| 5 | 契約チェック（パッケージのテストと同じ内容）を通す | notebook |
| 6 | `evaluate()` で組み込み metric と並べて実行する | notebook |
| 7 | `metrics/` に1ファイル置く | パッケージ |
| 8 | テスト・lint・CLI で確認する | ターミナル（notebook からも実行可） |
| 9 | config.yaml から使う | config |

### metric を書くときに守るルール

1. **時点をずらさない。** フォワードリターンの計算・ラグ・カレンダーの整列は pipeline の仕事です。
   metric では `data.aligned()` で取り出した同じ行をそのまま使い、`shift` などでずらしません。
2. **欠損は欠損のまま。** リターンやシグナルを `fillna(0)` しません。
3. **metric は `DataBundle` を受け取って `MetricResult` を返すだけ。** ファイルの読み書きや DB アクセスはしません。

## Step 1. 準備

リポジトリのルートで次を実行して notebook を開きます（VS Code の場合は `.venv` のカーネルを選びます）。

```sh
uv sync --group example
uv run --group example jupyter notebook
```

パッケージは editable インストールされているので、`src/` のコードを編集すると notebook にも反映されます。
`autoreload` を有効にしておくと、カーネルを再起動せずに反映されます。

In [1]:
%load_ext autoreload
%autoreload 2

from pathlib import Path

import pandas as pd

from alpha_signal_evaluation.data.columns import DATE
from alpha_signal_evaluation.testing import make_synthetic_bundle

# リポジトリのルート（pyproject.toml のあるディレクトリ）。Step 7 以降で使う
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "pyproject.toml").exists())
pd.set_option("display.width", 120)

## Step 2. 評価データ `DataBundle` を知る

metric が受け取るのは `DataBundle` 1つだけです。本番では pipeline が config.yaml に従ってファイルや DB から作りますが、
ここでは `make_synthetic_bundle()` で合成データを作ります（パッケージの契約テストでも使っているデータです）。

合成データのシグナル:

- `alpha`: 次の期のリターンを予測するように作ってある（効くシグナル）
- `noise`: リターンと無関係
- `mixed`: `alpha` と `noise` の平均

In [2]:
bundle = make_synthetic_bundle()

print("signals :", bundle.signal_names)
print("returns :", bundle.return_kinds)  # 系列名 -> 種類（total / specific）
print("horizons:", bundle.horizons)  # 単位はカレンダーの行数
print("dates   :", len(bundle.dates), "行", bundle.dates[:3], "...")

signals : ['alpha', 'noise', 'mixed']
returns : {'total': 'total', 'specific': 'specific'}
horizons: [1, 3]
dates   : 36 行 ('20150131', '20150228', '20150331') ...


主な属性:

| 属性 | 形 | 内容 |
|---|---|---|
| `signals` | index=(date, asset_id)、columns=シグナル名 | シグナル値 |
| `forward_returns[系列名][h]` | index=date、columns=asset_id | 行 t のシグナルに対応する、h 期先までのリターン（時点合わせ済み） |
| `return_kinds` | 系列名 → `total` / `specific` | リターンの種類 |
| `classification` | index=(date, asset_id)、columns=[category, ...] | 業種などの分類（config で指定した場合） |
| `risk_models` | 名前 → `RiskModelData` | エクスポージャー・ファクターリターン（config で指定した場合） |
| `dates` | tuple | 評価対象のカレンダー行（昇順） |

metric の中でよく使うメソッド:

| メソッド | 内容 |
|---|---|
| `aligned(signal, returns, h)` | シグナルとフォワードリターンを (date, asset_id) で結合する（どちらかが欠損の行は除く） |
| `signal_panel(signal)` | シグナルを date × asset_id の wide 形式にする |
| `resolve_signals(...)` / `resolve_returns(...)` / `resolve_horizons(...)` | パラメータが `None`（省略）なら「全部」に解決する |

In [3]:
bundle.signals.head()

alpha     noise     mixed
date     asset_id                              
20150131 A0000     0.125730  0.175003  0.150367
         A0001    -0.132105  0.641995  0.254945
         A0002     0.640423 -0.212498  0.213962
         A0003     0.104900       NaN  0.809940
         A0004    -0.535669 -1.707560 -1.121615

In [4]:
# total リターン・ホライズン 1 のフォワードリターン（wide 形式）
bundle.forward_returns["total"][1].iloc[:3, :5]

asset_id,A0000,A0001,A0002,A0003,A0004
date,,,,,
20150131,0.010234,0.015248,-0.002226,0.049751,-0.055477
20150228,-0.030334,-0.038742,-0.078110,-0.032102,-0.041794
20150331,0.006300,0.013382,-0.016519,-0.039337,0.082722


In [5]:
# metric で使うのはほぼこれ。シグナルとフォワードリターンが同じ行に並ぶ
bundle.aligned("alpha", "total", 1).head()

alpha  forward_return
date     asset_id                          
20150131 A0000     0.125730        0.010234
         A0001    -0.132105        0.015248
         A0002     0.640423       -0.002226
         A0003     0.104900        0.049751
         A0004    -0.535669       -0.055477

### 「時点合わせ済み」とは

`aligned()` の行 (t, i) には「t 時点の銘柄 i のシグナル」と「t 時点から h 期先までの銘柄 i のリターン」が並んでいます。
この対応は pipeline が保証しているので、metric は **同じ行をそのまま使うだけ** です。

自分でずらすと結果が壊れることを確かめてみます。

In [6]:
def mean_rank_ic(df: pd.DataFrame, signal: str) -> float:
    # 日付ごとの順位相関の平均（確認用の簡易版）
    ic = df.groupby(level=DATE).apply(
        lambda g: g[signal].corr(g["forward_return"], method="spearman")
    )
    return ic.mean()


ok = bundle.aligned("alpha", "specific", 1)

# やってはいけない例: フォワードリターンを1期ずらしてから結合する
fwd = bundle.forward_returns["specific"][1].shift(1)
bad = ok[["alpha"]].join(fwd.stack().rename("forward_return"), how="inner").dropna()

print(f"そのまま使う : {mean_rank_ic(ok, 'alpha'):.3f}")
print(f"1期ずらす    : {mean_rank_ic(bad, 'alpha'):.3f}")

そのまま使う : 0.301
1期ずらす    : -0.013


1期ずれただけで IC がほぼ 0 になりました。逆向きにずれればルックアヘッド（未来の情報を使った評価）になります。
どちらもエラーにならず、静かに結果を壊します。だから時点の操作は `pipeline/preprocess.py` だけで行い、metric では `shift` を使いません。

## Step 3. pandas で計算ロジックを試作する

まずはクラスのことは考えず、1つの組（シグナル・リターン系列・ホライズン）について計算を書きます。

`top_n_excess` の定義:

- 日付ごとに、シグナル上位 `n` 銘柄のフォワードリターンの等ウェイト平均 → `top_mean`
- 同じ日付の全銘柄（シグナルとリターンが揃っている銘柄）の平均 → `universe_mean`
- `excess = top_mean - universe_mean`
- 銘柄数が `min_obs` 未満の日付は欠損にする（0 にはしない）

In [7]:
signal, ret, h = "alpha", "total", 1
n, min_obs = 20, 40

df = bundle.aligned(signal, ret, h)  # index=(date, asset_id)、columns=[alpha, forward_return]
g = df.groupby(level=DATE)
rank = g[signal].rank(ascending=False, method="first")  # 1 = シグナル最大
top_mean = df["forward_return"].where(rank <= n).groupby(level=DATE).mean()
universe_mean = g["forward_return"].mean()
count = g.size()
excess = (top_mean - universe_mean).where(count >= min_obs)

pd.DataFrame(
    {"top_mean": top_mean, "universe_mean": universe_mean, "excess": excess, "n": count}
).head()

,top_mean,universe_mean,excess,n
date,,,,
20150131,0.013574,-0.013202,0.026776,75
20150228,0.016495,-0.002279,0.018774,77
20150331,0.008752,0.001008,0.007744,76
20150430,0.016150,0.009415,0.006735,77
20150531,-0.014685,-0.020253,0.005568,75


動いたら、後でクラスから呼べるように関数にまとめます。

ついでに、評価対象の全カレンダー行（`dates`）で reindex しておきます。データのない日付も欠損の行として残るので、
組み込み metric（`ic` など）の時系列と行が揃います。

In [8]:
def top_n_excess_timeseries(df: pd.DataFrame, signal: str, n: int, min_obs: int) -> pd.DataFrame:
    # 日付ごとの上位 n 銘柄の平均・全銘柄の平均・超過リターン・銘柄数
    g = df.groupby(level=DATE)
    rank = g[signal].rank(ascending=False, method="first")
    top_mean = df["forward_return"].where(rank <= n).groupby(level=DATE).mean()
    universe_mean = g["forward_return"].mean()
    count = g.size()
    return pd.DataFrame(
        {
            "top_mean": top_mean,
            "universe_mean": universe_mean,
            "excess": (top_mean - universe_mean).where(count >= min_obs),
            "n": count,
        }
    )


ts = top_n_excess_timeseries(bundle.aligned("alpha", "total", 1), "alpha", n=20, min_obs=40)
ts = ts.reindex(pd.Index(bundle.dates, name=DATE))
ts.tail()

,top_mean,universe_mean,excess,n
date,,,,
20170831,0.006154,-0.011019,0.017173,77
20170930,0.003193,-0.005763,0.008956,74
20171031,0.013488,0.001936,0.011551,74
20171130,-0.001819,-0.014524,0.012705,77
20171231,0.051292,0.017032,0.034261,76


### 答えがわかっているデータで確かめる

合成データでは `alpha` は効く・`noise` は効かないとわかっているので、結果がその通りになるかを見ます。
ロジックの誤り（符号の逆転、ランクの向きの間違いなど）はここで見つかります。

In [9]:
for s in bundle.signal_names:
    x = top_n_excess_timeseries(bundle.aligned(s, "total", 1), s, n=20, min_obs=40)["excess"]
    print(f"{s:>6}: 平均超過リターン = {x.mean():+.4f}")

 alpha: 平均超過リターン = +0.0122
 noise: 平均超過リターン = +0.0003
 mixed: 平均超過リターン = +0.0107


### 要約統計はパッケージの共通部品を使う

`metrics/_common.py` に、組み込み metric が使っている計算の部品があります。自前で書かずに使うと、組み込み metric と定義が揃います。

| 関数 | 内容 |
|---|---|
| `summarize_series(x, horizon)` | 時系列の平均・標準偏差・t 値・正の割合・期間数。t 値はホライズンの重なりを補正した Newey-West（lag = h − 1） |
| `cross_sectional_corr(df, x, y, method, min_obs)` | 日付ごとのクロスセクション相関（IC） |
| `assign_quantiles(values, n_quantiles)` | 日付ごとの分位 |
| `flatten_summary(frame, keys, stats)` | summary の表を `"{signal}/{returns}/h{horizon}/{stat}"` の辞書にする |
| `concat(frames)` | DataFrame のリストを縦に連結する（空リストなら空の DataFrame） |

In [10]:
from alpha_signal_evaluation.metrics._common import concat, flatten_summary, summarize_series

summarize_series(ts["excess"], horizon=1)

{'mean': 0.012160975832607565,
 'std': 0.00942773870258797,
 't_stat': 7.849271781265506,
 'hit_rate': 0.9444444444444444,
 'n_periods': 36.0}

## Step 4. `Metric` クラスにする

パッケージに組み込める形にします。書くのは次の5つです。

| 項目 | 内容 |
|---|---|
| `name` | config.yaml から参照する名前。他の metric と重複しないこと |
| `description` | `alpha-eval metrics list` に表示する1行の説明（既存の metric に合わせて英語） |
| `requires` | 使うデータの宣言。`signals` / `forward_returns` / `classification` / `risk_models` / `benchmark_weights` など |
| `Params` | パラメータ（pydantic のモデル）。**全項目にデフォルト値を付ける** |
| `compute(data)` | `DataBundle` を受け取り `MetricResult` を返す |

`MetricResult` は3つの入れ物を持ちます。

- `tables`: 表の名前 → DataFrame（時系列などの詳細）
- `summary`: キー → float（1つの数値で表せる要約。キーは `"{signal}/{returns}/h{horizon}/{stat}"` の形）
- `notes`: レポートに載せる注記（文字列のリスト）

### 4-1. パラメータ

`SignalReturnParams` を継承すると `signals` / `returns` / `horizons`（いずれも省略時は全部）が付きます。
存在しないシグナル名などを指定すると、計算を始める前にエラーになります（検証は自動で入ります）。

値の範囲は `Field(ge=...)`、項目どうしの関係は `model_validator` で書きます。
未知のキー（タイプミス）は自動で拒否されます。

In [11]:
from pydantic import Field, model_validator

from alpha_signal_evaluation.metrics import SignalReturnParams


class TopNExcessParams(SignalReturnParams):
    # signals / returns / horizons は SignalReturnParams が持っている
    n: int = Field(default=20, ge=1)  # 上位として扱う銘柄数
    min_obs: int = Field(default=40, ge=2)  # 日付ごとの最小銘柄数

    @model_validator(mode="after")
    def _min_obs_exceeds_n(self) -> "TopNExcessParams":
        if self.min_obs <= self.n:
            raise ValueError(f"min_obs ({self.min_obs}) must be greater than n ({self.n})")
        return self


print(TopNExcessParams())  # デフォルトだけで作れること

signals=None returns=None horizons=None n=20 min_obs=40


In [12]:
# 設定ミスはここで弾かれる（config.yaml の params もこのモデルで検証される）
for bad in [{"n": 0}, {"n": 30, "min_obs": 10}, {"top_n": 5}]:
    try:
        TopNExcessParams.model_validate(bad)
    except ValueError as e:
        print(f"{bad} -> {e.errors()[0]['msg']}")

{'n': 0} -> Input should be greater than or equal to 1
{'n': 30, 'min_obs': 10} -> Value error, min_obs (10) must be greater than n (30)
{'top_n': 5} -> Extra inputs are not permitted


### 4-2. `compute`

Step 3 の関数を、シグナル × リターン系列 × ホライズンのすべての組について呼びます。ループの形は組み込み metric（`ic.py` など）と同じです。

notebook ではまだ `@register_metric` を付けず、インスタンスを作って `compute` を直接呼びます（登録は Step 6）。

In [13]:
from alpha_signal_evaluation import DataBundle, Metric, MetricResult


class TopNExcess(Metric):
    name = "top_n_excess"
    description = "Excess forward return of the top-N signal assets over the cross-sectional mean"
    requires = frozenset({"signals", "forward_returns"})
    Params = TopNExcessParams

    def compute(self, data: DataBundle) -> MetricResult:
        p = self.params
        series, summary = [], []
        for signal in data.resolve_signals(p.signals):
            for ret in data.resolve_returns(p.returns):
                for h in data.resolve_horizons(p.horizons):
                    df = data.aligned(signal, ret, h)  # 時点合わせ済み。ずらさない
                    ts = top_n_excess_timeseries(df, signal, p.n, p.min_obs)
                    ts = ts.reindex(pd.Index(data.dates, name=DATE))
                    key = {"signal": signal, "returns": ret, "horizon": h}
                    series.append(ts.assign(**key).reset_index())
                    summary.append({**key, **summarize_series(ts["excess"], h)})

        summary = pd.DataFrame(summary).rename(columns={"mean": "mean_excess", "std": "std_excess"})
        return MetricResult(
            tables={"timeseries": concat(series), "summary": summary},
            summary=flatten_summary(
                summary, ["signal", "returns", "horizon"], ["mean_excess", "t_stat"]
            ),
            notes=["t_stat is Newey-West with lag = horizon - 1 (overlapping horizons)."],
        )

In [14]:
result = TopNExcess(n=20, min_obs=40).compute(bundle)
result.tables["summary"]

,signal,returns,horizon,mean_excess,std_excess,t_stat,hit_rate,n_periods
0,alpha,total,1,0.012161,0.009428,7.849272,0.944444,36.0
1,alpha,total,3,0.013660,0.014922,6.451029,0.777778,36.0
2,alpha,specific,1,0.011349,0.005551,12.440547,0.972222,36.0
3,alpha,specific,3,0.011932,0.012735,6.600584,0.805556,36.0
4,noise,total,1,0.000264,0.006579,0.244214,0.527778,36.0
5,noise,total,3,0.000399,0.010885,0.222143,0.416667,36.0
6,noise,specific,1,-0.000346,0.006331,-0.332832,0.500000,36.0
7,noise,specific,3,-0.001219,0.010128,-0.709943,0.416667,36.0
8,mixed,total,1,0.010689,0.006908,9.416033,1.000000,36.0
9,mixed,total,3,0.011926,0.012789,6.224942,0.861111,36.0


In [15]:
result.tables["timeseries"].head()

,date,top_mean,universe_mean,excess,n,signal,returns,horizon
0,20150131,0.013574,-0.013202,0.026776,75,alpha,total,1
1,20150228,0.016495,-0.002279,0.018774,77,alpha,total,1
2,20150331,0.008752,0.001008,0.007744,76,alpha,total,1
3,20150430,0.016150,0.009415,0.006735,77,alpha,total,1
4,20150531,-0.014685,-0.020253,0.005568,75,alpha,total,1


In [16]:
result.summary

{'alpha/total/h1/mean_excess': 0.012160975832607565,
 'alpha/total/h1/t_stat': 7.849271781265506,
 'alpha/total/h3/mean_excess': 0.013659626092727072,
 'alpha/total/h3/t_stat': 6.451028790471065,
 'alpha/specific/h1/mean_excess': 0.0113493937906319,
 'alpha/specific/h1/t_stat': 12.440546999794767,
 'alpha/specific/h3/mean_excess': 0.011932402286682745,
 'alpha/specific/h3/t_stat': 6.600583830611621,
 'noise/total/h1/mean_excess': 0.0002640158078300416,
 'noise/total/h1/t_stat': 0.2442135930441374,
 'noise/total/h3/mean_excess': 0.0003991188155266283,
 'noise/total/h3/t_stat': 0.22214304933452803,
 'noise/specific/h1/mean_excess': -0.0003463046028156709,
 'noise/specific/h1/t_stat': -0.3328316284213481,
 'noise/specific/h3/mean_excess': -0.0012189562218438259,
 'noise/specific/h3/t_stat': -0.7099434827048654,
 'mixed/total/h1/mean_excess': 0.01068880929647097,
 'mixed/total/h1/t_stat': 9.416033216450355,
 'mixed/total/h3/mean_excess': 0.01192647878443037,
 'mixed/total/h3/t_stat': 6.224

パラメータは次のどの形でも渡せます。config.yaml の `params` は2つ目の形（辞書）で渡されます。

```python
TopNExcess(n=10)
TopNExcess({"n": 10})
TopNExcess(TopNExcessParams(n=10))
```

## Step 5. 契約チェックを通す

パッケージの `tests/metrics/test_contract.py` は、登録されたすべての metric について次を自動で検証します。
notebook で同じことを確かめておけば、Step 8 のテストで落ちることはありません。

1. `requires` に宣言したデータだけで計算できる（宣言していないデータは空にして渡される）
2. `tables` か `summary` の少なくとも一方が空でない。`tables` の値は DataFrame、`summary` の値は float
3. `description` が空でない。`Params()` がデフォルトだけで作れる。未知のパラメータを拒否する
4. `check()` が、利用できるデータに対してエラーを返さない

よくある失敗:

- `classification` を使っているのに `requires` に書き忘れた → 1 で `None` が渡されて失敗する
- `summary` に int や文字列を入れた → 2 で失敗する（`float(...)` に変換する。`flatten_summary` は変換済み）
- デフォルト値のないパラメータがある → 3 で失敗する

In [17]:
from alpha_signal_evaluation.metrics import MetricParams


def check_contract(metric_cls, bundle) -> None:
    # tests/metrics/test_contract.py と同じ検証（notebook 用）
    # 1, 2: requires に宣言したデータだけで計算できる
    result = metric_cls().compute(bundle.restrict(metric_cls.requires))
    assert isinstance(result, MetricResult)
    assert result.tables or result.summary, "tables と summary が両方空"
    assert all(isinstance(t, pd.DataFrame) for t in result.tables.values())
    assert all(isinstance(v, float) for v in result.summary.values()), "summary の値は float"
    # 3: 宣言
    assert metric_cls.description, "description が空"
    assert issubclass(metric_cls.Params, MetricParams)
    metric_cls.Params()  # デフォルトだけで作れる
    try:
        metric_cls.Params.model_validate({"__unknown__": 1})
    except ValueError:
        pass
    else:
        raise AssertionError("未知のパラメータが拒否されない")
    metric_cls.Params.model_json_schema()  # CLI の metrics list で使う
    # 4: 計算前の検証
    errors = metric_cls.check(metric_cls.Params(), bundle.spec())
    assert errors == [], errors
    print(f"{metric_cls.name}: OK")


check_contract(TopNExcess, bundle)

top_n_excess: OK


## Step 6. `evaluate()` で組み込み metric と並べて実行する

`register_metric` でレジストリに登録すると、config の `metrics` から名前で呼べるようになります。
notebook で定義したクラスも、このカーネルの中でなら登録できます。

`evaluate(config, data=bundle)` のように `data` を渡すと、ファイルや DB を読まずにそのデータで評価します。

In [18]:
from alpha_signal_evaluation import evaluate, register_metric

register_metric(TopNExcess)

report = evaluate(
    {
        "version": 1,
        "metrics": [
            {"name": "ic", "params": {"returns": ["total"]}},
            {"name": "top_n_excess", "params": {"n": 20, "min_obs": 40, "returns": ["total"]}},
        ],
    },
    data=bundle,
    n_jobs=1,  # notebook で定義したクラスは別プロセスに渡せないので、逐次実行にする
)
report.status()

,metric,name,status,elapsed_sec,error
0,ic,ic,ok,0.018,
1,top_n_excess,top_n_excess,ok,0.013,


In [19]:
report.summary()

,metric,stat,value
0,ic,alpha/total/h1/mean_ic,0.280620
1,ic,alpha/total/h1/t_stat,11.913346
2,ic,alpha/total/h3/mean_ic,0.184523
3,ic,alpha/total/h3/t_stat,9.494950
4,ic,noise/total/h1/mean_ic,-0.002379
5,ic,noise/total/h1/t_stat,-0.116549
6,ic,noise/total/h3/mean_ic,0.014173
7,ic,noise/total/h3/t_stat,0.669071
8,ic,mixed/total/h1/mean_ic,0.200396
9,ic,mixed/total/h1/t_stat,10.398294


In [20]:
report["top_n_excess"].tables["summary"]

,signal,returns,horizon,mean_excess,std_excess,t_stat,hit_rate,n_periods
0,alpha,total,1,0.012161,0.009428,7.849272,0.944444,36.0
1,alpha,total,3,0.013660,0.014922,6.451029,0.777778,36.0
2,noise,total,1,0.000264,0.006579,0.244214,0.527778,36.0
3,noise,total,3,0.000399,0.010885,0.222143,0.416667,36.0
4,mixed,total,1,0.010689,0.006908,9.416033,1.000000,36.0
5,mixed,total,3,0.011926,0.012789,6.224942,0.861111,36.0


設定ミスは計算を始める前にまとめて検出されます。

In [21]:
from alpha_signal_evaluation import ConfigValidationError

try:
    evaluate(
        {
            "version": 1,
            "metrics": [
                {"name": "top_n_excess", "id": "typo", "params": {"signals": ["alhpa"]}},
                {"name": "top_n_excess", "id": "bad_n", "params": {"n": 50, "min_obs": 10}},
            ],
        },
        data=bundle,
    )
except ConfigValidationError as e:
    print(e)

Config validation failed:
  - metrics[typo]: unknown signals ['alhpa']; available: ['alpha', 'noise', 'mixed']
  - metrics[bad_n].params.(root): Value error, min_obs (10) must be greater than n (50)


## Step 7. パッケージに組み込む（`metrics/` に1ファイル置く）

notebook で形ができたら、`src/alpha_signal_evaluation/metrics/` に1ファイル置きます。**他のファイル（registry・config・CLI など）は変更不要** です。

1. `metrics/_template.py` をコピーして `metrics/top_n_excess.py` を作る。
   ファイル名は `_` で始めないこと（`_` で始まるモジュールは自動探索の対象外）。
2. Step 3〜4 のコードを移す。notebook との違いは次の3点。
   - import はパッケージ内の相対 import にする（`from ..data.bundle import DataBundle` など）
   - クラスに `@register_metric` を付ける（`metrics/` 直下のモジュールは自動で import され、その時点で登録される）
   - docstring（日本語）に「何をどう測るか」「パラメータの意味」「出力する表と列」「summary のキー」を書く

次のセルは `%%writefile` で実際にファイルを書き出します。ファイルの中身もこのセルで確認できます。
（チュートリアルとして試すだけの場合は、最後の「後片付け」でファイルを削除します。）

In [22]:
# このノートブックは notebooks/ をカレントディレクトリにして実行する前提
assert (Path.cwd() / "../src/alpha_signal_evaluation/metrics").is_dir(), (
    "notebooks/ で開いてください"
)

In [23]:
%%writefile ../src/alpha_signal_evaluation/metrics/top_n_excess.py
"""上位 N 銘柄の超過リターンの metric ``top_n_excess``。

日付ごとに、シグナル上位 N 銘柄のフォワードリターンの等ウェイト平均から全銘柄の平均を
引いた超過リターンを求め、その時系列を要約する。
"""

import pandas as pd
from pydantic import Field, model_validator

from ..data.bundle import DataBundle
from ..data.columns import DATE
from ._common import concat, flatten_summary, summarize_series
from .base import Metric, MetricResult, SignalReturnParams
from .registry import register_metric


class TopNExcessParams(SignalReturnParams):
    """``top_n_excess`` metric のパラメータ。

    Attributes:
        signals: 評価するシグナル名。デフォルト ``None``（全シグナル）。
        returns: 評価するリターン系列名。デフォルト ``None``（全系列）。
        horizons: 評価するホライズン。デフォルト ``None``（全ホライズン）。
        n: 上位として扱う銘柄数（1 以上）。デフォルト 20。
        min_obs: 日付ごとの最小銘柄数（``n`` より大きいこと）。デフォルト 40。
            未満の日付の超過リターンは欠損。
    """

    n: int = Field(default=20, ge=1)
    min_obs: int = Field(default=40, ge=2)

    @model_validator(mode="after")
    def _min_obs_exceeds_n(self) -> "TopNExcessParams":
        """``min_obs`` が ``n`` より大きいことを確認する（上位と全銘柄が同じにならないように）。

        Raises:
            ValueError: ``min_obs <= n`` の場合。
        """
        if self.min_obs <= self.n:
            raise ValueError(f"min_obs ({self.min_obs}) must be greater than n ({self.n})")
        return self


def top_n_excess_timeseries(df: pd.DataFrame, signal: str, n: int, min_obs: int) -> pd.DataFrame:
    """日付ごとに上位 n 銘柄の平均フォワードリターンと全銘柄の平均を求める。

    上位はシグナルの降順の順位（同順位は出現順）で決める。全銘柄はその日付に
    シグナルとフォワードリターンが揃っている銘柄。

    Args:
        df: ``DataBundle.aligned`` の出力（index=(date, asset_id)、
            columns=[signal, "forward_return"]）。
        signal: シグナルの列名。
        n: 上位として扱う銘柄数。
        min_obs: 超過リターンを計算する最小銘柄数。未満の日付の ``excess`` は欠損。

    Returns:
        index=date、columns=[top_mean, universe_mean, excess, n] の DataFrame。
        データのある日付だけが行になる。
    """
    g = df.groupby(level=DATE)
    rank = g[signal].rank(ascending=False, method="first")
    top_mean = df["forward_return"].where(rank <= n).groupby(level=DATE).mean()
    universe_mean = g["forward_return"].mean()
    count = g.size()
    return pd.DataFrame(
        {
            "top_mean": top_mean,
            "universe_mean": universe_mean,
            "excess": (top_mean - universe_mean).where(count >= min_obs),
            "n": count,
        }
    )


@register_metric
class TopNExcess(Metric):
    """上位 N 銘柄の超過リターン（上位 N 銘柄の平均 − 全銘柄の平均）。

    シグナル × リターン系列 × ホライズンの組ごとに、``data.aligned`` で取り出した
    同じ (date, asset_id) のシグナルとフォワードリターンについて、日付ごとに
    シグナル上位 ``n`` 銘柄の等ウェイト平均フォワードリターンから全銘柄の平均を引く。
    銘柄数が ``min_obs`` 未満の日付は欠損になる。h > 1 では h 期間のリターンのまま
    （1期間あたりに換算しない）。

    超過リターンの時系列から平均・標準偏差・t 値・正の割合・期間数を求める。t 値は
    ホライズンの重複による系列相関を補正するため lag = horizon − 1 の Newey-West。

    パラメータは :class:`TopNExcessParams` を参照。

    出力:
        tables["timeseries"]: 日付ごとの値。評価対象の全カレンダー行（``data.dates``）で
            reindex するため、データのない日付は欠損。
            columns=[date, top_mean, universe_mean, excess, n, signal, returns, horizon]。
            ``n`` はその日付の銘柄数。
        tables["summary"]: 1行が (signal, returns, horizon) の1組。
            columns=[signal, returns, horizon, mean_excess, std_excess, t_stat, hit_rate,
            n_periods]。
        summary: ``"{signal}/{returns}/h{horizon}/mean_excess"`` と
            ``"{signal}/{returns}/h{horizon}/t_stat"``。
    """

    name = "top_n_excess"
    description = "Excess forward return of the top-N signal assets over the cross-sectional mean"
    requires = frozenset({"signals", "forward_returns"})
    Params = TopNExcessParams

    def compute(self, data: DataBundle) -> MetricResult:
        """全ての (signal, returns, horizon) の組について超過リターンの時系列と要約を求める。

        Args:
            data: pipeline が用意した時点合わせ済みのデータ。

        Returns:
            ``timeseries`` / ``summary`` の表、平均超過リターンと t 値の summary、
            t 値の計算方法の注記を持つ MetricResult。
        """
        p = self.params
        series, summary = [], []
        for signal in data.resolve_signals(p.signals):
            for ret in data.resolve_returns(p.returns):
                for h in data.resolve_horizons(p.horizons):
                    df = data.aligned(signal, ret, h)
                    ts = top_n_excess_timeseries(df, signal, p.n, p.min_obs)
                    ts = ts.reindex(pd.Index(data.dates, name=DATE))
                    key = {"signal": signal, "returns": ret, "horizon": h}
                    series.append(ts.assign(**key).reset_index())
                    summary.append({**key, **summarize_series(ts["excess"], h)})

        summary = pd.DataFrame(summary).rename(columns={"mean": "mean_excess", "std": "std_excess"})
        return MetricResult(
            tables={"timeseries": concat(series), "summary": summary},
            summary=flatten_summary(
                summary, ["signal", "returns", "horizon"], ["mean_excess", "t_stat"]
            ),
            notes=["t_stat is Newey-West with lag = horizon - 1 (overlapping horizons)."],
        )

Writing ../src/alpha_signal_evaluation/metrics/top_n_excess.py


### パッケージ版を notebook で使う

このカーネルでは Step 6 で notebook 版の `TopNExcess` を `top_n_excess` という名前で登録済みです。
同じ名前を別のクラスで登録しようとするとエラーになります（名前の重複を防ぐ仕組み）。

In [24]:
import importlib

try:
    importlib.import_module("alpha_signal_evaluation.metrics.top_n_excess")
except ValueError as e:
    print(e)

Metric name 'top_n_excess' is already registered by __main__.TopNExcess; cannot register alpha_signal_evaluation.metrics.top_n_excess.TopNExcess


通常は **カーネルを再起動** すれば、パッケージ版が自動で登録されます。
ここでは再起動せずに続けるため、notebook 版の登録を外してからパッケージ版を import します
（notebook で試すとき専用の操作です。パッケージのコードではやりません）。

In [25]:
from alpha_signal_evaluation.metrics.registry import _REGISTRY

_REGISTRY.pop("top_n_excess", None)
pkg = importlib.import_module("alpha_signal_evaluation.metrics.top_n_excess")
check_contract(pkg.TopNExcess, bundle)

top_n_excess: OK


パッケージ版のクラスは別プロセスからも import できるので、並列実行（`n_jobs > 1`）も使えます。

In [26]:
report = evaluate(
    {
        "version": 1,
        "metrics": [
            {"name": "ic"},
            {"name": "quantile"},
            {"name": "top_n_excess", "params": {"n": 20}},
        ],
    },
    data=bundle,
    n_jobs=2,
)
report.status()[["metric", "status", "error"]]

,metric,status,error
0,ic,ok,
1,quantile,ok,
2,top_n_excess,ok,


In [27]:
report["top_n_excess"].tables["summary"]

,signal,returns,horizon,mean_excess,std_excess,t_stat,hit_rate,n_periods
0,alpha,total,1,0.012161,0.009428,7.849272,0.944444,36.0
1,alpha,total,3,0.013660,0.014922,6.451029,0.777778,36.0
2,alpha,specific,1,0.011349,0.005551,12.440547,0.972222,36.0
3,alpha,specific,3,0.011932,0.012735,6.600584,0.805556,36.0
4,noise,total,1,0.000264,0.006579,0.244214,0.527778,36.0
5,noise,total,3,0.000399,0.010885,0.222143,0.416667,36.0
6,noise,specific,1,-0.000346,0.006331,-0.332832,0.500000,36.0
7,noise,specific,3,-0.001219,0.010128,-0.709943,0.416667,36.0
8,mixed,total,1,0.010689,0.006908,9.416033,1.000000,36.0
9,mixed,total,3,0.011926,0.012789,6.224942,0.861111,36.0


この後は `src/alpha_signal_evaluation/metrics/top_n_excess.py` をエディタで直接編集して開発を続けられます。
`autoreload` が有効なので、保存すれば次のセルの実行から反映されます。

## Step 8. テスト・lint・CLI で確認する

ターミナルでリポジトリのルートから実行します（notebook からは先頭に `!` を付けて実行できます）。

```sh
uv run pytest tests/metrics -k top_n_excess   # 契約テスト（置いたファイルは自動でテスト対象になる）
uv run pytest                                 # 全体
uv run ruff check src tests && uv run ruff format src tests
uv run alpha-eval metrics list -v             # 登録されているか・パラメータの一覧
```

notebook で書いたコードには、自分の metric 専用のテスト（「答えがわかっているデータで確かめる」で書いたような確認）を
`tests/metrics/` に足しておくと、後から変更したときに安心です。

In [28]:
!cd "{ROOT}" && uv run pytest tests/metrics -q -k top_n_excess

...                                                                      [100%]
3 passed, 41 deselected in 0.34s


In [29]:
!cd "{ROOT}" && uv run ruff check src/alpha_signal_evaluation/metrics/top_n_excess.py && uv run ruff format --check src/alpha_signal_evaluation/metrics/top_n_excess.py

All checks passed!
1 file already formatted


In [30]:
!cd "{ROOT}" && uv run alpha-eval metrics list -v | grep -A6 "^top_n_excess"

top_n_excess  Excess forward return of the top-N signal assets over the cross-sectional mean  (requires: forward_returns, signals)
    signals: {"anyOf": [{"items": {"type": "string"}, "type": "array"}, {"type": "null"}], "default": null, "title": "Signals"}
    returns: {"anyOf": [{"items": {"type": "string"}, "type": "array"}, {"type": "null"}], "default": null, "title": "Returns"}
    horizons: {"anyOf": [{"items": {"type": "integer"}, "type": "array"}, {"type": "null"}], "default": null, "title": "Horizons"}
    n: {"default": 20, "minimum": 1, "title": "N", "type": "integer"}
    min_obs: {"default": 40, "minimum": 2, "title": "Min Obs", "type": "integer"}


## Step 9. config.yaml から使う

config.yaml の `metrics` に追加するだけです。

```yaml
metrics:
  - name: ic
  - name: top_n_excess
    params: {n: 20, min_obs: 40, returns: [specific], horizons: [1, 3]}
  - name: top_n_excess          # 同じ metric を別の設定で使うときは id が必須
    id: top_n_excess_n50
    params: {n: 50, min_obs: 100}
```

```sh
uv run alpha-eval validate config.yaml   # パラメータの誤り（存在しないシグナル名など）は計算前に検出される
uv run alpha-eval run config.yaml        # 結果は output.dir/top_n_excess/summary.csv などに書き出される
```

Python からは次のように使います。

```python
from alpha_signal_evaluation import evaluate

report = evaluate("config.yaml")
report["top_n_excess"].tables["summary"]
```

## 付録 A. 自分のデータで開発する

合成データで形ができたら、実データで確かめます。config.yaml がある場合は、pipeline に `DataBundle` を作らせます
（時点合わせは本番と同じ処理です）。

```python
from alpha_signal_evaluation import load_config, validate
from alpha_signal_evaluation.pipeline.preprocess import build_bundle

cfg = load_config("path/to/config.yaml")
validate(cfg).raise_for_errors()       # 先に設定とファイルの存在を確認する
prepared = build_bundle(cfg)
real = prepared.bundle
print(prepared.warnings)

TopNExcess(n=50, min_obs=200).compute(real).tables["summary"]
```

手元の DataFrame から作る場合は `make_bundle` を使います（期間リターンからフォワードリターンを計算するところまで行います）。

```python
from alpha_signal_evaluation import make_bundle

real = make_bundle(
    signals,                                   # index=(date, asset_id)、columns=シグナル名
    {"total": total_ret, "specific": spec_ret}, # 期間リターン（index=date、columns=asset_id）
    return_kinds={"total": "total", "specific": "specific"},
    horizons=[1, 3],
)
```

サンプルの config で試すこともできます。

```sh
uv run python examples/example1/make_sample_data.py   # examples/example1/input にサンプルデータを作る
```

```python
cfg = load_config(ROOT / "examples/example1/config.example.yaml")
```

## 付録 B. シグナルとリターン以外のデータを使う

使うデータは必ず `requires` に書きます。書いていないデータを使うと、契約テストで `None` や空のデータが渡されて失敗します。

| `requires` に書く名前 | 使い方 |
|---|---|
| `classification` | `data.classification["category"]`（index=(date, asset_id)）。`aligned()` の結果に `join` して使う |
| `risk_models` | `name, model = data.risk_model(p.risk_model)`。`model.exposures`、`model.factors("style")`、`model.forward_factor_returns[h]` |
| `risk_models.exposures` など | リスクモデルの一部だけを使う場合 |
| `benchmark_weights` | `data.benchmark_weights["weight"]`（index=(date, asset_id)） |

パラメータに `risk_model: str | None = None` を持たせると、存在しないリスクモデル名も計算前に検出されます。

### データに依存する検証を足す

`Params` だけでは判断できない検証（例: 指定した分類の列が config に存在するか）は、`check` を上書きして足します。
`check` はデータを読み込む前に呼ばれ、エラーメッセージのリストを返します。

```python
@classmethod
def check(cls, params, spec):
    errors = super().check(params, spec)   # requires・signals・returns・horizons の検証
    if ...:
        errors.append("...")
    return errors
```

### 累積リターン

期間をまたぐリターンの集計（算術平均と幾何平均、kind ごとの累積方法）は `ReturnAggregation` を使うと、組み込みの `quantile` と同じ規約になります。

## 付録 C. 本体のリポジトリを触らずに配布する

自分のパッケージに metric を置き、entry point で登録することもできます。
自分のパッケージの `pyproject.toml` に次のように書くと、`alpha_signal_evaluation` の起動時に読み込まれます。

```toml
[project.entry-points."alpha_signal_evaluation.metrics"]
top_n_excess = "my_package.top_n_excess"
```

この場合、import は相対 import ではなく `from alpha_signal_evaluation.metrics import Metric, MetricResult, register_metric` のようにします。
テストには `alpha_signal_evaluation.testing.make_synthetic_bundle()` が使えます。

## チェックリスト

- [ ] `shift` や `fillna(0)` を使っていない（時点合わせは pipeline、欠損は欠損のまま）
- [ ] 使うデータをすべて `requires` に書いた
- [ ] `Params` の全項目にデフォルト値がある。値の範囲は `Field`、項目どうしの関係は `model_validator` で検証している
- [ ] `summary` の値は float、キーは `"{signal}/{returns}/h{horizon}/{stat}"` の形
- [ ] docstring（日本語）に、何を測るか・パラメータ・出力する表と列・summary のキーを書いた
- [ ] `uv run pytest` と `uv run ruff check src tests` / `uv run ruff format src tests` が通る

## 後片付け

チュートリアルとして試しただけなら、Step 7 で作ったファイルを削除します。
**自分の metric を本当に組み込む場合は、このセルを実行しないでください。**

In [31]:
(ROOT / "src/alpha_signal_evaluation/metrics/top_n_excess.py").unlink(missing_ok=True)